# Metabolic Forest — product door (`xenosite.forest`)

This notebook walks through the **Rust chematic** public API (PyO3 handles, not dicts).
Examples use realistic drug-like substrates from the crate gold tests.

**Prerequisites:** extension built (`uv sync` or `maturin develop --features python,extension-module`).
See [docs/forest/RUST.md](../docs/forest/RUST.md).

In [ ]:
from xenosite.forest import (
    ForestMol,
    MetabolicNetwork,
    available,
    default_ruleset,
    find_path,
    find_path_partial,
    normalize_tautomer,
    phase_one,
    product_graph_bfs,
    product_graph_ruleset,
    random_path,
)

assert available(), "Build xenosite.forest._rust (maturin develop)"
print("extension OK")

## 1. `find_path` — acetaminophen toward NAPQI

Acetaminophen (APAP) bioactivation to **NAPQI** is a classic Phase I tox pathway
(hydroxylation / quinone-imine chemistry). We search with **`default_ruleset()`**
(includes tautomerization) or **`phase_one()`** for a Phase I–only catalog.

In [ ]:
APAP = "CC(=O)Nc1ccc(O)cc1"
NAPQI = "CC(=O)N=C1C=CC(=O)C=C1"

hits, counters = find_path(
    APAP,
    NAPQI,
    max_paths=2,
    max_nodes=80,
    use_atom_diff=False,
)
print(f"hits={len(hits)} billed={counters.billed} nodes={counters.to_dict()['nodes']}")
for i, hit in enumerate(hits):
    d = hit.to_dict()
    print(f"[{i}] product CSMI: {hit.smiles}")
    for step in d["steps"]:
        print(f"    {step['rule']} @ {step['site']}")

**Handles:** `hit.mol` is a tagged `ForestMol`; `hit.plan` is a `StepPlan` (elementary deps).
Export-only JSON: `hit.to_dict()`.

In [ ]:
if hits:
    h = hits[0]
    print("plan length:", len(h.plan))
    print("tagged mol CSMI:", h.mol.csmi)

## 2. `find_path` — terbinafine N-dealkylation

Terbinafine loses the **N-methyl–naphthyl** arm to give an aldehyde metabolite (TBA).
Gold test uses `phase_one()` and a stereo-specific target spelling from chematic.

In [ ]:
TERBINAFINE = "CN(C/C=C/C#CC(C)(C)C)Cc1cccc2ccccc12"
TBF_ALDEHYDE = r"C(#C/C=C/C=O)C(C)(C)C"

hits, counters = find_path(
    TERBINAFINE,
    TBF_ALDEHYDE,
    max_paths=1,
    max_nodes=40,
    use_atom_diff=False,
)
assert hits, "expected dealkylation path"
print(hits[0].to_dict()["steps"][0])

## 3. `find_path_partial` — exact hits plus closest misses

When the target is unreachable within `max_nodes`, partial outcomes carry a **residual**
(atom-diff style gap) for the closest products on the graph.

In [ ]:
exact, partials, counters = find_path_partial(
    APAP,
    NAPQI,
    max_paths=1,
    max_nodes=80,
    use_atom_diff=False,
)
print(f"exact={len(exact)} partial={len(partials)}")
if partials:
    p = partials[0].to_dict()
    print("closest:", p["smiles"], "residual cost:", p["residual"]["cost"])

## 4. One-step **`metabolize`** on APAP (Phase I catalog)

`RuleSet.metabolize` returns **`Emission`** handles and accepts the public
`ForestMol` wrapper or the extension pyclass.

In [ ]:
mol = ForestMol(APAP)
emissions = phase_one().metabolize(mol)
print(f"{len(emissions)} emissions from phase_one()")
for e in emissions[:8]:
    prods = e.product_csmis()
    print(f"  {e.pattern_name} / {e.rule_path} -> {prods[:2]}{'…' if len(prods) > 2 else ''}")

## 5. BFS **`product_graph_bfs`** (bounded exploration)

**Warning:** default BFS can run a long time. Always set `max_nodes`, `max_depth`, and
prefer **`target=`** when exploring toward one product.

Default rules: **`product_graph_ruleset()`** (QuinoneFormation + EpoxideHydration + Phase I core,
**no** Tautomerization).

In [ ]:
EUGENOL = "COc1ccc(O)cc1"
QUINONE_LIKE = "O=C1C=C(O)C(=O)C(O)=C1"

net = product_graph_bfs(
    EUGENOL,
    target=QUINONE_LIKE,
    max_nodes=64,
    max_depth=4,
)
print("nodes:", net.n_nodes(), "edges:", net.n_edges(), "reaches target:", net.reaches(QUINONE_LIKE))
print("ruleset default name:", product_graph_ruleset().name)

## 6. One **`MetabolicNetwork`** for BFS + search

Pass the same `network=` handle to **`product_graph_bfs`** and **`find_path`** so BFS layers
and search hops accumulate on one graph. Then **`step_plan_between`** summarizes routes
between node indices.

In [ ]:
net = MetabolicNetwork()
product_graph_bfs("CC", target="CCO", max_nodes=32, max_depth=3, network=net)
hits, _ = find_path("CC", "CCO", max_paths=1, max_nodes=200, network=net)
root = net.root_idx()
target_i = net.index_of(hits[0].smiles)
plan = net.step_plan_between(root, target_i)
print("ethanol index:", target_i, "plan steps:", len(plan))
print("parents of ethanol:", net.parents(hits[0].smiles))

## 6b. **`GraphNode` / `GraphEdge`** views

Index the live graph without marshalling: ``net[i]`` is a node view; ``inbound_edge(slot)``
exposes rule, parent index, and tagged ``kept_mol``.

In [ ]:
from xenosite.forest import GraphEdge, GraphNode  # noqa: F401 — re-exported pyclasses

eth = net[target_i]
edge = eth.inbound_edge(0)
print(edge.rule, "parent:", edge.parent_index, "kept:", edge.kept_mol.csmi)

## 7. Tautomer normalization (chematic pick)

Enol/keto pairs can be aligned before search with `normalize_tautomer=True` on `find_path`.
Here we normalize reactants directly.

In [ ]:
keto, _ = normalize_tautomer("CC=O")
enol, changed = normalize_tautomer("OC=C")
print("enol changed:", changed, "same CSMI:", enol.csmi == keto.csmi)

## 8. Seeded **`random_path`** (pathway sampling)

Explores one stochastic walk under Phase I rules — useful for priors and fuzz tests.

In [ ]:
walk = random_path(APAP, seed=42, max_steps=3)
print("path:", walk.path)
print(walk.to_dict())

## 9. Optional — RDKit drawing

Install `xenosite-forest[rdkit]`. **`to_rdkit()`** is a CSMI view; atom tags are not round-tripped.

In [ ]:
try:
    from rdkit.Chem import Draw

    m = ForestMol(APAP).to_rdkit()
    Draw.MolToImage(m, size=(320, 240))
except ImportError:
    print("Skip: pip install xenosite-forest[rdkit]")

## 10. Rulesets at a glance

| Factory | Use |
|---------|-----|
| `default_ruleset()` | `find_path` default (includes Tautomerization) |
| `phase_one()` | Flat Phase I catalog (QF + EH + leaves) |
| `product_graph_ruleset()` | BFS default (QF + EH + Phase I core, no tautomer) |

In [ ]:
for rs in (default_ruleset(), phase_one(), product_graph_ruleset()):
    print(rs.name, "top-level members:", len(rs))